# TrialMatch: Colab A100 setup and feasibility

Selected target: **Qwen2.5-1.5B-Instruct**. Draft: **Qwen2.5-0.5B-Instruct**.

This notebook checks the environment, tokenizers, both models, one LoRA optimizer step,
and adapter save/reload. It is **not full training or a final evaluation**. No test examples
are displayed or scored. The smoke adapter is not a final checkpoint.

**Before running:** choose **Runtime → Change runtime type → GPU → A100** when available.
Upload the companion `trialmatch-colab-starter.zip` in the next cell. Run cells in order.
Keep model caches on Colab's local disk; persist the small logs and adapter to Drive.

Educational screening research prototype; not a clinical enrollment decision.


## 1. Upload and unpack the project

The archive is provided with this notebook; no GitHub repository is needed yet.
This cell accepts the expected starter ZIP and extracts it into a new project directory.
If rerunning in an existing session, it reuses the existing project; start a fresh runtime
to upload a different version. Do not upload credentials or patient records.


In [ ]:
from pathlib import Path
import os
import stat
import zipfile

PROJECT = Path('/content/trialmatch')
if not (PROJECT / 'pyproject.toml').exists():
    from google.colab import files
    uploaded = files.upload()
    expected = 'trialmatch-colab-starter.zip'
    if expected not in uploaded:
        raise ValueError(f'Upload {expected}.')
    archive_path = Path('/content') / expected
    archive_path.write_bytes(uploaded[expected])
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for entry in archive.infolist():
            destination = (PROJECT / entry.filename).resolve()
            if not destination.is_relative_to(PROJECT.resolve()):
                raise ValueError('Archive contains an unsafe path')
            if stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError('Archive contains a symbolic link')
        archive.extractall(PROJECT)
    del uploaded
os.chdir(PROJECT)
os.environ['HF_HOME'] = '/content/trialmatch-hf-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Project:', PROJECT)


## 2. Install the pinned starter dependencies

Preserve Colab's preinstalled CUDA PyTorch. The proposed Transformers/PEFT stack is
pinned separately, and the actual runtime versions will be saved with the run.
If Colab asks for a runtime restart, restart it and rerun from cell 1; the project remains
on disk for that session. Do not use “Disconnect and delete runtime” during a run.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt',
                '-e', '.'], check=True)
# Check our runtime stack in a fresh subprocess so imports do not stay stale in the notebook.
subprocess.run([sys.executable, '-c',
    'import torch, transformers, peft; '
    'print("torch", torch.__version__, "transformers", transformers.__version__, "peft", peft.__version__); '
    'assert torch.cuda.is_available(), "Select a GPU runtime"; '
    'print(torch.cuda.get_device_name(0))'], check=True)


## 3. Choose persistent output storage

Recommended: mount your Drive and keep `USE_GOOGLE_DRIVE = True`. Colab will request
access interactively. Only generated run artifacts go into the folder below; model weights
stay in the temporary local cache. Set this to False to use temporary storage, then download
outputs before disconnecting.


In [ ]:
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/TrialMatch/results')
else:
    OUTPUT_ROOT = PROJECT / 'results/colab'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Outputs:', OUTPUT_ROOT)


## 4. Recreate the exact dataset split and run integrity tests

The dataset commit comes from the supplied manifest, not the changing `main` branch.
This recreates all split files mechanically. Subsequent smoke code reads **train only**.
Do not open `test.jsonl` or use it for development. All original sentence IDs and expert
label categories are preserved; generated GPT annotations are excluded.


In [ ]:
import json

manifest = json.loads((PROJECT / 'data/manifests/split_manifest.json').read_text())
subprocess.run([sys.executable, 'scripts/prepare_data.py', '--revision',
                manifest['dataset_revision']], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## 5. Run the model and one-batch feasibility checks

This downloads both selected model checkpoints. On first use their immutable revisions are
resolved and recorded. For later experiments, copy the successful run's `model_revisions.json`
to the project's `configs/model_revisions.json` and keep using those commits.

The script checks vocabulary/token IDs, special tokens, chat templates and encoding probes;
loads both models; runs synthetic greedy generation; rejects overlength training examples
without truncation; then performs exactly **one optimizer step on one training example**.
It saves/reloads the adapter and checks synthetic greedy-token equality.

BF16 is used when supported; otherwise FP32 is used and recorded. The draft is unloaded
before the training check. This is not an inference benchmark and does not implement the
assignment's custom decoders yet. A setup failure writes `status.json` and `failure.txt`.


In [ ]:
from datetime import datetime, timezone

RUN_DIR = OUTPUT_ROOT / ('feasibility-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RUN_DIR.mkdir(parents=True, exist_ok=False)
command = [sys.executable, '-u', 'scripts/colab_feasibility.py',
           '--output-dir', str(RUN_DIR), '--max-length', '2048']
with (RUN_DIR / 'console.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f'Feasibility failed. Inspect {RUN_DIR}/status.json and console.log')


## 6. Review the results

Success means this small configuration ran correctly. It does not establish clinical accuracy,
full-dataset memory requirements, training convergence or speculative-decoding speedups.
Save these artifacts for the project log and share them before choosing full training settings.


In [ ]:
for filename in ['status.json', 'environment.json', 'tokenizer_compatibility.json',
                 'train_preparation.json', 'one_step.json', 'adapter_reload.json']:
    print('\n' + filename)
    print((RUN_DIR / filename).read_text())


## 7. Optional: download the run artifacts

The archive includes the small smoke adapter and logs, not the downloaded base weights.
If Drive is enabled, the original output folder is already persistent. Downloading provides
an additional portable copy. Keep this one-step adapter separate from final checkpoints.


In [ ]:
DOWNLOAD_RESULTS = True
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files
    result_zip = shutil.make_archive('/content/' + RUN_DIR.name, 'zip', root_dir=RUN_DIR)
    files.download(result_zip)


## Next stage

After these checks pass: freeze model revisions, review training/validation token lengths,
implement structured-output evaluation on validation, establish the untouched base-model
baseline, and implement full supervised adapter training. Keep all final test evaluation
until the configuration is frozen. See `docs/next-steps.md`.
